# `torch.linalg.eig`

`torch.linalg.eig` takes a matrix input and decomposes it into eigenvalues and eigenvectors. The goal of this note is to understand how it can be implemented to run on a GPU. It is meant to be somewhat high to mid level, to give a fairly concise explanation which can be understood as long as you know some basic linear algebra and PyTorch. So I will purposefully leave out some details.

## Recap of the eigenvalue problem

A square matrix $A$ describes a linear operator, which can be applied to a vector $u$ using matrix-vector multiplication to produce a new transformed vector $A u$.

In general $A u \ne a u$, where $a$ is a scalar. However, for every linear operator there is a set of nonzero vectors $v$, called eigenvectors, which do scale by some scalar value $a$, called an eigenvalue, when the operator is applied to them.

$$
A v = a v
\tag 1
$$

The goal of the eigenvalue problem is: given some operator $A$, find every eigenvalue $a$ that satisfies equation (1) and also find one eigenvector $v$ for each eigenvalue. An $n \times n$ matrix will have $n$ eigenvalues, though some may actually have the same value. The choice of eigenvectors is not unique, so it is often desirable to normalize them and also choose orthogonal eigenvectors if possible.

### Example: Characteristic equation

Let's say we're given the operator:

$$
A = \begin{pmatrix}
1 & 3 & 1 \\
0 & 2 & 0 \\
0 & 1 & 4
\end{pmatrix}
$$

One way to solve the eigenvalue problem by hand is to analytically solve what is called the characteristic equation or [characteristic polynomial](https://en.wikipedia.org/wiki/Characteristic_polynomial) of the matrix. This method is very limited because it only works for matrices of size $4 \times 4$ or smaller. An $n \times n$ matrix will produce an $n$-the degree polynomial, and only polynomials of degree 4 or less have analytic solutions in general. But it's easy to follow as an example, so it's worth taking a look at it. The characteristic equation is:

$$
\det(A - I a) = 0
$$

We just have to plug in $A$ and simplify to produce a set of equations to find the three eigenvalues.

$$
\det
\begin{pmatrix}
1 - a & 3 & 1 \\
0 & 2 - a & 0 \\
0 & 1 & 4 -a
\end{pmatrix}
= (1 - a) (2 - a) (4 - a) = 0
$$

The eigenvalues are:

$$
a_1 = 1 \quad
a_2 = 2 \quad
a_3 = 4
$$

Now we can find the eigenvectors individually.

$$
Av_1 = a_1 v_1
= \begin{pmatrix}
v_{11} + 3 v_{12} + v_{13} \\
2 v_{12} \\
v_{12} + 4 v_{13}
\end{pmatrix}
= \begin{pmatrix}
v_{11} \\ v_{12} \\ v_{13}
\end{pmatrix}
\rightarrow\quad
\begin{pmatrix}
3 v_{12} + v_{13} \\
v_{12} \\
v_{12} + 3 v_{13}
\end{pmatrix}
= 0
\quad\rightarrow\quad
\begin{matrix}
v_{11} = \text{arbitrary} \\
v_{12} = 0 \\
v_{13} = 0
\end{matrix}
$$

Since we want normalized eigenvectors, $v_{11} = 1$ is a good choice.

$$
v_1 = \begin{pmatrix}
1 \\ 0 \\ 0
\end{pmatrix}
$$

I won't write out the solution for the other two eigenvectors, because it's just more of the same. Let's see what `torch.linalg.eig` gives for this matrix.

In [1]:
import torch

r = torch.linalg.eig(torch.tensor([
    [1, 3, 1],
    [0, 2, 0],
    [0, 1, 4],
], dtype=torch.float))

# Note: `torch.linalg.eig` returns the individual eigenvectors as columns of a
# matrix, so we have to transpose.
for a, v in zip(r.eigenvalues, r.eigenvectors.T):
    print(f'eigenvalue: {a.item()}, eigenvector: {v}')

eigenvalue: (1+0j), eigenvector: tensor([1.+0.j, 0.+0.j, 0.+0.j])
eigenvalue: (4+0j), eigenvector: tensor([0.3162+0.j, 0.0000+0.j, 0.9487+0.j])
eigenvalue: (2+0j), eigenvector: tensor([ 0.9129+0.j,  0.3651+0.j, -0.1826+0.j])


All the eigenvalues match what we found above, and the eigenvector for $a_1 = 1$ also matches what we found.

## The QR algorithm

Since the characteristic equation method won't work in general, we need a more robust way to solve the eigenvalue problem. One of the most popular methods is called the [QR algorithm](https://en.wikipedia.org/wiki/QR_algorithm), and it is the method that `torch.linalg.eig` uses.

### QR iteration

The heart of the algorithm is called QR iteration. The first iteration starts with $A_1 = A$ and computes further iterations like so:

$$
Q_k, R_k = \text{decompQR}(A_k)
$$
$$
A_{k+1} = R_k @ Q_k
$$

$\text{decompQR}$ represents the [QR decomposition](https://en.wikipedia.org/wiki/QR_decomposition) of a matrix. I won't go into the details of it.

After $n$ iterations, the hope is that $A_n$ will converge toward a triangular matrix. If that happens, then the diagonal elements of $A_n$ will be the eigenvalues of the original matrix $A$.

We'll check for convergence by taking the absolute value sum of each of the lower diagonal columns and compare them to zero. Once they're close enough to zero, we'll consider the matrix to be approximately triangular. This convergence check is based on the [Gershgorin circle theorem](https://en.wikipedia.org/wiki/Gershgorin_circle_theorem).


Let's try out QR iteration on the example matrix from earlier.

In [2]:
def is_approx_triangular(A, atol=1e-6, rtol=1e-6):
    lower_diag_sum = A.tril(diagonal=-1).abs().sum(dim=1)
    return torch.allclose(
        lower_diag_sum,
        torch.zeros_like(lower_diag_sum),
        atol=atol,
        rtol=rtol
    )

# Returns (result, num iterations run, whether it converged)
def QR_iteration_naive(A, max_num_iters=1000):
    if not A.is_complex():
        # Convert to complex to allow for complex eigenvalues
        A = torch.complex(A, torch.zeros_like(A))

    for i in range(max_num_iters):
        Q, R = torch.linalg.qr(A)
        A = R @ Q

        if is_approx_triangular(A):
            return A, i + 1, True

    return A, max_num_iters, False

In [3]:
A = torch.tensor([
    [1, 3, 1],
    [0, 2, 0],
    [0, 1, 4],
], dtype=torch.float)

QR_iteration_naive(A)

(tensor([[ 1.0000e+00+0.j,  1.0000e+00+0.j, -3.0000e+00+0.j],
         [ 0.0000e+00+0.j,  4.0000e+00+0.j, -1.0000e+00+0.j],
         [ 0.0000e+00+0.j,  9.5368e-07+0.j,  2.0000e+00+0.j]]),
 22,
 True)

It took 22 iterations to converge on an approximately triangular matrix. And look, we've obtained the correct eigenvalues on the diagonal!

But convergence is not guaranteed for all matrices. Here's an example which will never converge:

In [4]:
A = torch.tensor([
    [0, -1],
    [1, 0],
], dtype=torch.float)

QR_iteration_naive(A)

(tensor([[ 0.+0.j, -1.+0.j],
         [ 1.+0.j,  0.+0.j]]),
 1000,
 False)

This matrix has complex eigenvalues $\pm i$, and basic QR iteration doesn't converge if there are two complex eigenvalues with the same norm. So we'll need to modify the algorithm in some way to account for that.

## Resources

Lecture notes from [Numerical Methods for Solving Large Scale Eigenvalue Problems
(Spring semester 2018)](https://people.inf.ethz.ch/~arbenz/ewp/) by Prof. Dr. Peter Arbenz. Specifically [lecture 4 on the QR algorithm](https://people.inf.ethz.ch/arbenz/ewp/Lnotes/chapter4.pdf)

<https://en.wikipedia.org/wiki/QR_algorithm>